# oxDNA: Persistence Length Optimization

This notebook optimizes oxDNA1 force-field parameters to match a target persistence length using DiffTRe (Differentiable Trajectory Reweighting). Multiple parallel oxDNA simulations are coordinated via Ray.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

Here we demonstrate persistence length optimization: we run multiple oxDNA simulations in parallel, compute a persistence length observable from the combined trajectory, and optimize force-field parameters so the measured persistence length converges to a target value.

## Imports

In [ ]:
import logging
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_md
import mythos.energy.dna1 as dna1_energy
import mythos.optimization.objective as jdna_objective
import mythos.optimization.optimization as jdna_optimization
import optax
import ray
from mythos.input import oxdna_input
from mythos.observables import base
from mythos.observables.persistence_length import PersistenceLength
from mythos.simulators import oxdna
from mythos.ui.loggers.console import ConsoleLogger

jax.config.update("jax_enable_x64", True)
logging.basicConfig(level=logging.INFO)
logging.getLogger("jax").setLevel(logging.WARNING)

## Configuration

In [ ]:
TARGET_LP = 40.0  # target persistence length in nm
NUM_SIMS = 10
LEARNING_RATE = 1e-3
OPT_STEPS = 100
INPUT_DIR = Path("../../data/templates/simple-helix-60bp-oxdna1").resolve()
OXDNA_SRC = Path("../../../oxDNA").resolve()

## Initialize Ray

The coordination of objectives and simulators is done through Ray tasks, so we
initialize a Ray server. Explicit initialization is required to pass the
environment variable `JAX_ENABLE_X64=true` to enable 64-bit precision in JAX,
which is important for numerical stability in Mythos optimizations. Ray worker
processes could also be started with this environment variable set which would
make explicit initialization unnecessary here.

In [ ]:
ray.init(
    log_to_driver=True,
    runtime_env={"env_vars": {"JAX_ENABLE_X64": "True"}},
)

## Load topology and build energy function

In [ ]:
inp = oxdna_input.read_input_dir(INPUT_DIR)

energy_fn = dna1_energy.create_default_energy_fn(
    topology=inp.topology,
    displacement_fn=jax_md.space.periodic(inp.box_size)[0],
).with_noopt("ss_stack_weights", "ss_hb_weights", "kt"
).with_params(kt=inp.kT)

transform_fn = energy_fn.energy_fns[0].transform_fn
opt_params = energy_fn.opt_params()

## Create parallel simulators

We create multiple `oxDNASimulator` instances — each with a unique name. The `RayOptimizer` will coordinate running them in parallel via Ray.

In [ ]:
simulators = oxdna.oxDNASimulator.create_n(
    NUM_SIMS,
    input_dir=INPUT_DIR,
    energy_fn=energy_fn,
    source_path=OXDNA_SRC,
)

## Define the persistence length objective

The `PersistenceLength` observable measures persistence length from the trajectory. We wrap it in a DiffTRe objective that computes gradients via trajectory reweighting.

In [ ]:
lp_fn = PersistenceLength(
    rigid_body_transform_fn=transform_fn,
    displacement_fn=jax_md.space.periodic(inp.box_size)[0],
    quartets=base.get_duplex_quartets(int(inp.topology.n_nucleotides / 2)),
    truncate=40,
)


def lp_loss_fn(traj, weights, energy_model, *args, **kwargs):
    fit_lp = lp_fn(traj, weights=weights)
    loss = jnp.sqrt((fit_lp - TARGET_LP) ** 2)
    return loss, (("persistence_length", fit_lp), {})


required_obs = [name for sim in simulators for name in sim.exposes()]

persistence_length_objective = jdna_objective.DiffTReObjective(
    name="persistence_length",
    required_observables=required_obs,
    logging_observables=["loss", "persistence_length", "neff"],
    grad_or_loss_fn=lp_loss_fn,
    energy_fn=energy_fn,
    min_n_eff_factor=0.95,
    n_equilibration_steps=0,
    max_valid_opt_steps=10,
)

## Run the optimization

We use `RayOptimizer` to coordinate the parallel simulators. The `optimizer.run()` method manages the optimization loop internally.

In [ ]:
opt = jdna_optimization.RayOptimizer(
    objectives=[persistence_length_objective],
    simulators=simulators,
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    aggregate_grad_fn=lambda grads: grads[0],
    logger=ConsoleLogger(),
)

output = opt.run(params=opt_params, n_steps=OPT_STEPS)
print("\nOptimization complete!")
print(f"Final params: {output.opt_params}")